# 06 · Analysis: hypotheses H1–H6, BatchNorm controls, figures
### Machine Unlearning is Extinction, Not Forgetting

CPU-only notebook: it reads the result files of notebooks 01–05, runs the pre-registered tests, and saves
tables and figures.

| Test | Statistic per model (unlearned, matched) | Test |
|---|---|---|
| **H1** spontaneous recovery | primary score after the primary proxy minus before, **minus the same for the retrained model** | one-sided Wilcoxon signed-rank, > 0 |
| **H2a** renewal (ABC) | mean over new contexts of (context − clean) primary score, minus the retrained model's | Wilcoxon, > 0 |
| **H2b** renewal (ABA) | colour (A) minus gray (B) primary score of ABA models, minus the retrained model's | Wilcoxon, > 0 |
| **H3** reinstatement | recovery after *sibling* cues minus after *dissimilar* cues (within model), minus the retrained model's | Wilcoxon, > 0; plus a dose-response mixed model |
| **H4** savings | area under the relearning curve, unlearned minus retrained | Wilcoxon, > 0 |
| **H5** hidden features | linear-probe forget accuracy, unlearned minus retrained | Wilcoxon, > 0; Spearman correlation with H1 |
| **H5b** output selectivity | 2 × \|AUROC − 0.5\| of the forget-class output before any test (does that output still single out the class, in either direction?), unlearned minus retrained | Wilcoxon, > 0 |
| **H6** multi-context remedy | standard minus multi-context model (same method, class, seed): renewal on held-out contexts, and H1 recovery | Wilcoxon, > 0 (positive = the remedy reduces relapse) |
| **C1** BatchNorm artifact | recovery from recomputing BN statistics only (no weight change) | Wilcoxon, > 0 (positive = the BatchNorm illusion exists here) |
| **C2** BatchNorm share | retain fine-tuning recovery scored with updated minus original BN statistics | Wilcoxon, > 0 |
| **R** robustness | H1–H3 with the top-1 and rank scores; H3 and H4 scored with the original BN statistics | Wilcoxon, > 0 |
| bonus: collateral | retraining gap on siblings minus on unrelated classes | Wilcoxon, > 0 |

**Holm correction** is applied within each family: *main* (H1–H6), *control* (C1, C2, collateral) and
*robustness* (R). Effect sizes come with **bootstrap 95 % confidence intervals**. One unit = one (class, seed)
model; the pilot class is excluded.

**Scores (ext_utils 1.2).** The primary score is `top5_cc`: the share of forget-class test images with the
forgotten class among the model's top 5 answers, minus the same share on all other images. It is graded, so
partial recovery counts. The strict top-1 version (`chance_corrected`) has a floor: in the first pilot (run #4)
every value was exactly 0. `rank_score` (where the class ranks, 0..1) and `forget_auc` (how well its output
separates its images from the rest) are reported as robustness checks. Each hypothesis is tested on the
**difference to the retrained model** of the same class and seed, because graded scores can move for any model
that is fine-tuned or shown corrupted images; only the part a never-trained model cannot produce counts.

**BatchNorm and the primary tests.** Kalani et al. (2026) showed that recomputing BatchNorm statistics alone can
reverse apparent forgetting. This is controlled three ways: **C1** measures recovery from recomputed statistics
with no weight change at all; every fine-tuned model is also scored with its **original** statistics put back
(robustness rows); and the GroupNorm arm has no running statistics. The primary tests use the statistics the
fine-tuning produced (`update`), because scoring heavily fine-tuned weights with the old statistics breaks the
network: in pilot run #5 all 33 collapsed rows were of that kind. For the GroupNorm arm only `update` exists.


## Kaggle settings for this notebook

Set these in the right-hand **Settings** panel (in the editor: *View → Show sidebar* if it is hidden).

| Setting | Value | Why |
|---|---|---|
| **Accelerator** | **None** | This notebook only does statistics and plots on the CPU; leaving the GPU off saves your weekly GPU quota. |
| **Internet** | **On** | Needed to download CIFAR-100 (~170 MB) and, in notebook 04, CIFAR-100-C, unless you attach them as datasets. Internet requires a phone-verified Kaggle account. |
| **Persistence** | Files only (optional) | Keeps `/kaggle/working` between *interactive* sessions, so you don't lose files if the browser disconnects. Not needed for background runs. |
| **Environment** | *Always use latest* for the first run, then **Pin to original environment** | Pinning freezes the library versions (PyTorch etc.), so a Kaggle image update can't break a half-finished experiment. |
| **Language** | Python | |

**Inputs to attach** (*Add Input* → *Your Work* / *Your Datasets*):

| Input | Why |
|---|---|
| `ext-code` dataset (contains `ext_utils.py`) | The shared library. Upload it once as a private dataset. |
| Outputs of **notebooks 01–05** (`ext-01-originals`, `ext-02-retrain`, `ext-03-unlearning`, `ext-04-extinction`, `ext-05-mechanism`) | The result files (`results/*.jsonl`); checkpoints are not needed. |

**How to run.** Click **Save Version → Save & Run All (Commit)**. The notebook then runs in the background
for up to 12 hours with the browser closed. When it finishes, its files appear under *Output*, and the next
notebook attaches them as an input. 

**If a run stops early** (time budget or error), nothing is lost. Every finished job is written to disk
immediately. Attach this notebook's own latest output as an input and run it again: finished jobs are
skipped and their files are carried over.

## Setup: import the shared library

This cell finds `ext_utils.py` and imports it as `U`. All the real code (data pipeline, model, unlearning
methods, tests, job runner) lives there, so every notebook uses *identical* code.

`U.env_report()` then prints:
- the Python and PyTorch versions,
- which GPUs are visible (you should see **two Tesla T4** with the T4 x2 accelerator),
- free disk space,
- whether the internet is reachable,
- which inputs are attached.

**Check this output before going further.** If it says `GPU: none found`, the accelerator is off. If it
says `internet: OFF` and no CIFAR-100 dataset is attached, data loading will fail.

In [ ]:
# Locate ext_utils.py (from an attached dataset, a utility script, or the working folder) and import it.
import os, sys, glob
candidates = (glob.glob("/kaggle/working/ext_utils.py")
              + glob.glob("/kaggle/usr/lib/**/ext_utils.py", recursive=True)
              + glob.glob("/kaggle/input/**/ext_utils.py", recursive=True)
              + glob.glob(os.path.join(os.getcwd(), "ext_utils.py"))
              + glob.glob(os.path.join(os.getcwd(), "..", "ext_utils.py")))
if not candidates:
    raise FileNotFoundError(
        "ext_utils.py not found. Upload it as a private Kaggle Dataset (e.g. 'ext-code') and attach it "
        "with 'Add Input' -> 'Your Datasets' (see the Kaggle settings section above).")
sys.path.insert(0, os.path.dirname(os.path.abspath(candidates[0])))
import ext_utils as U
print("ext_utils loaded from:", U.UTILS_FILE)
U.env_report()

## Configuration

| Parameter | Value | What it does and why this value |
|---|---|---|
| `smoke` | `False` | Must match the other notebooks (it picks the `_smoke` or the real files). |
| `norm`, `seeds`, `forget_classes` | `"bn"`, `[0, 1, 2]`, 10 classes | Which arm to analyse. Must equal notebook 01 **of the same arm** (checked automatically). Main arm: leave as is. GroupNorm-arm copy (`ext-06-analysis-gn`): `norm="gn"`, `seeds=[0]`, first 5 classes. |
| `mctx_contexts` | `["clean", "gray", "blur"]` | Must equal notebook 03's setting: these contexts were seen during multi-context unlearning, so H6 measures renewal only on the *other* (held-out) contexts. |
| `MATCHED_ONLY` | `True` | Analyse only unlearned models that met the matching rule. These are the fair comparison, because their output-level forgetting equals retraining. Set `False` for a robustness check. |
| `ALPHA` | 0.05 | Significance level after Holm correction. |
| `N_BOOT` | 10000 | Bootstrap resamples for the confidence intervals. 10k gives stable 95 % bounds in seconds. |
| `PRIMARY_SR` | from `analysis_choices.json` | The primary spontaneous-recovery proxy, fixed during the pilot (notebook 04). If the file is missing, the default is continued retain training for `sr_epochs` epochs. |
| `BN_PRIMARY` | automatic | `"frozen"` when those rows exist (BatchNorm arm), otherwise `"update"` (GroupNorm arm). |
| `SAVINGS_SHOTS` | 20 | Shot count shown in the savings figure (all shot counts enter the test). |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os, json, warnings
from scipy import stats
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

CFG = U.make_cfg(
    smoke=False,
    norm="bn",                      # "gn" in the GroupNorm-arm copy (ext-06-analysis-gn)
    seeds=[0, 1, 2],                # GroupNorm arm: [0]
    forget_classes=["maple_tree", "tiger", "rose", "shark", "bus",
                    "apple", "chair", "butterfly", "girl", "castle"],   # GroupNorm arm: first 5 only
    mctx_contexts=["clean", "gray", "blur"],
)
U.check_experiment_config(CFG)
MATCHED_ONLY = True
ALPHA = 0.05
N_BOOT = 10000
SAVINGS_SHOTS = 20
choices = U.load_choices(CFG)
PRIMARY_SR = (choices.get("primary_sr_proxy", "retain_ft"), float(choices.get("primary_sr_level", CFG["sr_epochs"])))
print("primary spontaneous-recovery proxy:", PRIMARY_SR)
METRIC = choices.get("primary_metric", U.PRIMARY_METRIC)          # fixed on the pilot, before the main data
# forget_auc is not a response score (it drops toward 0.5 when an inverted selectivity fades, as for NegGrad+
# in pilot run #5); it is used for H5b instead.
ROBUST_METRICS = [m for m in U.RESPONSE_METRICS if m not in (METRIC, "forget_auc")]
print("primary score:", METRIC, "| robustness scores:", ROBUST_METRICS)
os.makedirs(U.FIG_DIR, exist_ok=True)
TAB_DIR = os.path.join(U.WORK, "tables"); os.makedirs(TAB_DIR, exist_ok=True)

## Load all results

`U.load_df(task, CFG)` gathers the rows of a task from every attached output, removing duplicates.
Unlearned, ABA and multi-context models are then filtered to matched models (if `MATCHED_ONLY`), the
pilot class is removed, and fine-tuning rows marked `collapsed` (the network diverged) are dropped.

In [ ]:
unl = U.load_df("unlearn", CFG)
ext = U.load_df("extinction", CFG)
mech = U.load_df("mechanism", CFG)
for name, d in [("unlearn", unl), ("extinction", ext), ("mechanism", mech)]:
    print(f"{name:11s}: {len(d)} rows")
IDS = ["kind", "method", "cls", "seed"]
PILOT = CFG["pilot_class"]

def keep(d):
    if d.empty: return d
    d = d[d["cls"] != PILOT]
    if MATCHED_ONLY and "matched" in d:
        d = d[~d["kind"].isin(["unl", "aba", "mctx"]) | (d["matched"] == True)]
    if "collapsed" in d:
        bad = d["collapsed"].fillna(False).astype(bool)
        if bad.any():
            print(f"excluding {int(bad.sum())} collapsed fine-tuning rows:",
                  d[bad].groupby(["test", "method"]).size().to_dict())
        d = d[~bad]
    return d

unl_main = unl[(unl["kind"] == "unl") & (unl["cls"] != PILOT)] if len(unl) else unl
if len(unl):
    print("matched rate per kind and method:")
    print(unl[unl["cls"] != PILOT].groupby(["kind", "method"])["matched"].mean().round(3).to_string())
ext, mech = keep(ext), keep(mech)
if len(ext) and METRIC not in ext:
    raise ValueError(f"Extinction results have no '{METRIC}' column. Attach notebook 04 output made with ext_utils 1.2.")
METHODS = [m for m in CFG["methods"] if len(ext) and m in set(ext["method"])]
BN_PRIMARY = "update"          # statistics produced by the fine-tuning (see the introduction)
BN_ROBUST = "frozen" if len(ext) and "bn" in ext and (ext["bn"] == "frozen").any() else None
print("methods with extinction results:", METHODS, "| BN mode for primary fine-tuning tests:", BN_PRIMARY,
      "| robustness:", BN_ROBUST)

## Helper functions

- `wilcoxon_greater`: one-sided Wilcoxon signed-rank test that the median of the values is > 0. It returns
  NaN (no test) when there are fewer than 2 values or all are 0.
- `boot_ci`: percentile bootstrap 95 % CI of the mean.
- `record`: stores one test (family, method, n, mean, CI, raw p) for the Holm correction at the end.
- `base_val`: each model's score before any manipulation.
- `minus_retrain`: a model's value minus the retrained model's value for the same class and seed (the
  hypothesis tests use this difference).
- `paired`: turns per-model values into standard-vs-other differences for the same method, class and seed.

In [ ]:
RESULTS = []

def wilcoxon_greater(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2 or np.allclose(x, 0):
        return np.nan
    try:
        return float(stats.wilcoxon(x, alternative="greater").pvalue)
    except ValueError:
        return np.nan

def boot_ci(x, n=N_BOOT, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) == 0: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    means = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return tuple(np.percentile(means, [2.5, 97.5]))

def record(hyp, method, values, family="main", note=""):
    v = np.asarray(values, float); v = v[~np.isnan(v)]
    lo, hi = boot_ci(v)
    RESULTS.append(dict(family=family, hypothesis=hyp, method=method, n=len(v), mean=v.mean() if len(v) else np.nan,
                        ci_low=lo, ci_high=hi, p_raw=wilcoxon_greater(v), note=note))

def base_val(d, metric=None):
    metric = metric or METRIC
    return d[d["test"] == "base"][IDS + [metric]].rename(columns={metric: "base"})

def minus_retrain(d, value, keys=()):
    """Value of each non-retrained model minus the retrained model's value (same class, seed and keys)."""
    keys = ["cls", "seed", *keys]
    if d.empty or "kind" not in d:
        return pd.DataFrame(columns=list(d.columns) + ["excess"])
    ref = d[d["kind"] == "retrain"][keys + [value]].rename(columns={value: "ref"})
    out = d[d["kind"] != "retrain"].merge(ref, on=keys, how="inner")
    return out.assign(excess=out[value] - out["ref"])

def paired(df, value, a="unl", b="mctx"):
    """a-minus-b differences of `value` for models of the same method, class and seed."""
    w = df[df["kind"].isin([a, b])].pivot_table(index=["method", "cls", "seed"], columns="kind", values=value)
    if a not in w or b not in w: return pd.DataFrame(columns=["method", "diff"])
    w = w.dropna(subset=[a, b]).reset_index()
    w["diff"] = w[a] - w[b]
    return w

## H1: spontaneous recovery

Recovery = primary score after the manipulation minus before it. The **primary** test uses the pre-registered
proxy, using `update` rows for fine-tuning and `eval` rows for noise/quantization/pruning (never `bn_recal`),
and tests the unlearned model's recovery minus the retrained model's. The other three scores are robustness
checks. All proxies and BN modes are tabulated as secondary results.


In [ ]:
prim = pd.DataFrame()

def h1(metric):
    s = ext[ext["test"] == "spontaneous"].merge(base_val(ext, metric), on=IDS)
    s["recovery"] = s[metric] - s["base"]
    ms = s[s["bn"].isin([BN_PRIMARY, "eval"])]
    pr = ms[(ms["proxy"] == PRIMARY_SR[0]) & np.isclose(ms["level"].astype(float), PRIMARY_SR[1])]
    return s, ms, pr

if len(ext) and (ext["test"] == "spontaneous").any():
    sr, main_sr, prim = h1(METRIC)
    e1 = minus_retrain(prim, "recovery")
    for m in METHODS:
        record("H1 spontaneous recovery", m, e1[(e1.kind == "unl") & (e1.method == m)]["excess"])
    for mt in ROBUST_METRICS:
        e = minus_retrain(h1(mt)[2], "recovery")
        for m in METHODS:
            record(f"R: H1 [{mt}]", m, e[(e.kind == "unl") & (e.method == m)]["excess"], family="robustness")
    print("retrained reference, mean recovery:", round(prim[prim.kind == "retrain"]["recovery"].mean(), 4))
    sr_table = sr[sr.kind.isin(["unl", "retrain"])].pivot_table(index=["proxy", "bn", "level"], columns="method", values="recovery", aggfunc="mean")
    sr_table.to_csv(os.path.join(TAB_DIR, "H1_spontaneous_all_proxies.csv"))
    display(sr_table.round(3))


## BatchNorm controls (C1, C2)

- **C1**: recovery after recomputing BatchNorm statistics from the largest number of retain images, with no
  weight change. A positive value means part of any "recovery" can come from normalization statistics alone.
- **C2**: retain fine-tuning recovery with statistics updating minus frozen, i.e. the share of fine-tuning
  recovery that comes from BatchNorm statistics.

These tell you how large the artifact is. They don't affect H1–H6, which already exclude it.

In [ ]:
if len(ext) and "sr" in dir() and (sr["proxy"] == "bn_recal").any():
    bnr = sr[(sr.proxy == "bn_recal") & sr.kind.isin(["unl", "retrain"])]
    top = bnr[bnr.level == bnr.level.max()]
    e_top = minus_retrain(top, "recovery")
    for m in METHODS:
        record("C1 BatchNorm recalibration alone", m, e_top[(e_top.kind == "unl") & (e_top.method == m)]["excess"], family="control")
    print("Recovery from BatchNorm recalibration only (rows: number of retain images):")
    display(bnr.pivot_table(index="level", columns="method", values="recovery", aggfunc="mean").round(3))
    rf = sr[sr.proxy == "retain_ft"]
    rf = rf[rf.level == rf.level.max()]
    w = rf.pivot_table(index=IDS, columns="bn", values="recovery").reset_index()
    if {"update", "frozen"} <= set(w.columns):
        w["bn_share"] = w["update"] - w["frozen"]
        for m in METHODS:
            record("C2 BN share of retain-FT recovery (update - frozen)", m, w[(w.kind == "unl") & (w.method == m)]["bn_share"], family="control")
        display(w.groupby(["kind", "method"])[["update", "frozen", "bn_share"]].mean().round(3))
else:
    print("No BatchNorm-recalibration rows (GroupNorm arm, or notebook 04 not run yet).")

## H2: renewal

**H2a (ABC):** for each unlearned model, the mean over all new contexts (synthetic and CIFAR-100-C) of
context-minus-clean primary score, minus the same for the retrained model.

**H2b (ABA):** for ABA models, colour (A) minus gray (B), minus the same for the retrained model. The same
quantity for normally unlearned models ("gray minus colour") is printed as a comparison.


In [ ]:
ctx = pd.DataFrame()
if len(ext) and (ext["test"] == "context").any():
    B = CFG["aba_context"]

    def h2(metric):
        c = ext[ext["test"] == "context"].merge(base_val(ext, metric), on=IDS)
        c["renewal"] = c[metric] - c["base"]
        per = c[c.kind.isin(["unl", "retrain"])].groupby(IDS)["renewal"].mean().reset_index()
        ab = c[c.kind.isin(["aba", "retrain"]) & (c.context == B)].copy()
        ab["A_minus_B"] = ab["base"] - ab[metric]                 # base = colour (A), context B = gray
        return c, minus_retrain(per, "renewal"), minus_retrain(ab, "A_minus_B")

    ctx, e2a, e2b = h2(METRIC)
    for m in METHODS:
        record("H2a renewal (ABC)", m, e2a[(e2a.kind == "unl") & (e2a.method == m)]["excess"])
        record("H2b renewal (ABA)", m, e2b[(e2b.kind == "aba") & (e2b.method == m)]["excess"])
    for mt in ROBUST_METRICS:
        _, a_, b_ = h2(mt)
        for m in METHODS:
            record(f"R: H2a [{mt}]", m, a_[(a_.kind == "unl") & (a_.method == m)]["excess"], family="robustness")
            record(f"R: H2b [{mt}]", m, b_[(b_.kind == "aba") & (b_.method == m)]["excess"], family="robustness")
    ctx_table = ctx[ctx.kind.isin(["unl", "retrain", "original"])].pivot_table(
        index=["source", "context", "severity"], columns="method", values="renewal", aggfunc="mean")
    ctx_table.to_csv(os.path.join(TAB_DIR, "H2a_renewal_by_context.csv"))
    display(ctx_table.round(3))
    std = ctx[(ctx.kind == "unl") & (ctx.context == B)]
    print("comparison - normally unlearned models, gray minus colour:",
          std.groupby("method")["renewal"].mean().round(3).to_dict())


## H3: reinstatement

Recovery after the last reinstatement epoch, for each cue condition. The primary test uses the **paired**
difference *sibling minus dissimilar* within each model, minus the same difference for the retrained model;
the version scored with the original BN statistics and the other scores are robustness checks. The dose-response analysis relates recovery to the graded relatedness score (the
original model's mean probability for the cue classes on forget-class images), using a mixed model with a
random intercept per class.

In [ ]:
import statsmodels.formula.api as smf

def reinstatement(bn_mode, metric=None):
    metric = metric or METRIC
    r = ext[(ext["test"] == "reinstatement") & (ext["bn"] == bn_mode)]
    if r.empty: return r
    r0 = r[r.level == 0][IDS + ["condition", metric]].rename(columns={metric: "v0"})
    r = r[r.level == r.level.max()].merge(r0, on=IDS + ["condition"])
    r["recovery"] = r[metric] - r["v0"]
    return r

def sib_minus_dis(r):
    if r.empty: return pd.DataFrame(columns=IDS + ["sib_minus_dis"])
    w = r.pivot_table(index=IDS, columns="condition", values="recovery").reset_index()
    if {"sibling", "dissimilar"} <= set(w.columns):
        w["sib_minus_dis"] = w["sibling"] - w["dissimilar"]
        return w
    return pd.DataFrame(columns=IDS + ["sib_minus_dis"])

ri = pd.DataFrame()
if len(ext) and (ext["test"] == "reinstatement").any():
    ri = reinstatement(BN_PRIMARY)
    e3 = minus_retrain(sib_minus_dis(ri), "sib_minus_dis")
    for m in METHODS:
        record("H3 reinstatement (sibling - dissimilar)", m, e3[(e3.kind == "unl") & (e3.method == m)]["excess"])
    if BN_ROBUST:
        e_fr = minus_retrain(sib_minus_dis(reinstatement(BN_ROBUST)), "sib_minus_dis")
        for m in METHODS:
            record("R: H3 with original BN statistics", m, e_fr[(e_fr.kind == "unl") & (e_fr.method == m)]["excess"], family="robustness")
    for mt in ROBUST_METRICS:
        e = minus_retrain(sib_minus_dis(reinstatement(BN_PRIMARY, mt)), "sib_minus_dis")
        for m in METHODS:
            record(f"R: H3 [{mt}]", m, e[(e.kind == "unl") & (e.method == m)]["excess"], family="robustness")
    display(ri[ri.kind.isin(["unl", "retrain"])].pivot_table(index="condition", columns="method", values="recovery", aggfunc="mean").round(3))
    dr = ri[ri.kind == "unl"].copy()
    if dr["cls"].nunique() >= 3 and len(dr) >= 12:
        try:
            fit = smf.mixedlm("recovery ~ relatedness + C(method)", dr, groups=dr["cls"]).fit()
            print(fit.summary().tables[1])
        except Exception as e:
            print("mixed model could not be fitted:", e)
    else:
        print("dose-response mixed model skipped (needs >= 3 classes).")
    if len(dr) > 2:
        rho = stats.spearmanr(dr["relatedness"], dr["recovery"])
        print(f"Spearman(relatedness, recovery) = {rho.correlation:.3f}, p = {rho.pvalue:.3g}")


## H4: rapid reacquisition (savings)

For each model and shot count: the area under the forget-accuracy relearning curve (normalised to 0–1). The
excess AUC of an unlearned model over the retrained model of the same class and seed is tested (pooled over
shot counts, averaged per model). Primary with the updated BN statistics; the original statistics as a robustness check. This is closely
related to the Anamnesis Index (Chundawat et al. 2023), which should be cited alongside it.

In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz      # NumPy 2 renamed trapz -> trapezoid

def auc(g):
    g = g.sort_values("level"); x = g["level"].astype(float).values; y = g["forget_test_acc"].values
    return float(_trapz(y, x) / (x.max() - x.min())) if len(x) > 1 and x.max() > x.min() else float(y.mean())

def savings_excess(bn_mode):
    sv = ext[(ext["test"] == "savings") & (ext["bn"] == bn_mode)]
    if sv.empty: return pd.DataFrame(columns=IDS + ["excess"]), sv
    a = pd.DataFrame([dict(zip(IDS + ["shots"], k), auc=auc(g)) for k, g in sv.groupby(IDS + ["shots"])])
    ref = a[a.kind == "retrain"][["cls", "seed", "shots", "auc"]].rename(columns={"auc": "auc_ref"})
    a = a[a.kind.isin(["unl", "mctx"])].merge(ref, on=["cls", "seed", "shots"])
    a["excess"] = a["auc"] - a["auc_ref"]
    return a, sv

if len(ext) and (ext["test"] == "savings").any():
    a, sv_primary = savings_excess(BN_PRIMARY)
    per_model = a.groupby(IDS)["excess"].mean().reset_index() if len(a) else a
    for m in METHODS:
        record("H4 savings (excess AUC)", m, per_model[(per_model.kind == "unl") & (per_model.method == m)]["excess"])
    if BN_ROBUST:
        a_fr, _ = savings_excess(BN_ROBUST)
        pm_fr = a_fr.groupby(IDS)["excess"].mean().reset_index() if len(a_fr) else a_fr
        for m in METHODS:
            record("R: H4 with original BN statistics", m, pm_fr[(pm_fr.kind == "unl") & (pm_fr.method == m)]["excess"], family="robustness")
    if len(a):
        display(a[a.kind == "unl"].pivot_table(index="shots", columns="method", values="excess", aggfunc="mean").round(3))

## H5: hidden knowledge in the features and in the output

**H5b** needs no probe: before any test, how well does the forget-class *output* still separate that class's
images from all others? Selectivity = 2 × |AUROC − 0.5| (0 = no information, 1 = perfect, in either direction:
NegGrad+ in pilot run #5 gave the forgotten class its *lowest* output on exactly its own images, AUROC 0.025).
The retrained model never learned the class, so its value is the baseline (0.05 in the pilot).


Excess probe accuracy = linear-probe forget-class accuracy of the unlearned model minus that of the
retrained model (same class and seed). We also check whether models with more hidden knowledge also recover
more (Spearman correlation with the H1 recovery). Probes and head swaps are already common in the 2026
literature (e.g. Gao et al., Jang et al.), so H5 is supporting evidence for the mechanism, not a headline claim.

In [ ]:
if len(ext) and (ext["test"] == "base").any():
    b = ext[ext["test"] == "base"].copy()
    b["selectivity"] = (b["forget_auc"] - 0.5).abs() * 2
    e5b = minus_retrain(b[b.kind.isin(["unl", "retrain"])], "selectivity")
    for m in METHODS:
        record("H5b output selectivity", m, e5b[(e5b.kind == "unl") & (e5b.method == m)]["excess"])
    display(b.groupby(["kind", "method"])[["forget_auc", "selectivity"]].mean().round(3))

if len(mech) and (mech["test"] == "probe").any():
    pr = mech[mech["test"] == "probe"]
    ref = pr[pr.kind == "retrain"][["cls", "seed", "probe_forget_test_acc"]].rename(columns={"probe_forget_test_acc": "probe_ref"})
    pu = pr[pr.kind == "unl"].merge(ref, on=["cls", "seed"])
    pu["probe_excess"] = pu["probe_forget_test_acc"] - pu["probe_ref"]
    for m in METHODS:
        record("H5 probe excess", m, pu[pu.method == m]["probe_excess"])
    display(pr.groupby(["kind", "method"])[["probe_forget_test_acc", "probe_test_acc"]].mean().round(3))
    if len(prim):
        j = pu.merge(prim[prim.kind == "unl"][IDS + ["recovery"]], on=IDS)
        if len(j) > 2:
            r = stats.spearmanr(j["probe_excess"], j["recovery"])
            print(f"Spearman(probe excess, H1 recovery) = {r.correlation:.3f}, p = {r.pvalue:.3g}, n = {len(j)}")

## H6: does multi-context unlearning reduce relapse?

Each multi-context model is paired with the standard unlearned model of the same method, class and seed.
The difference (standard − multi-context) is positive when multi-context unlearning relapses **less**.
- **Renewal** uses only **held-out** contexts: every CIFAR-100-C corruption, plus the synthetic contexts that
  were not part of `mctx_contexts`.
- **Spontaneous recovery** uses the primary H1 proxy.
- **Reinstatement** uses recovery after sibling cues.

In [ ]:
if len(ext) and (ext["kind"] == "mctx").any():
    train_ctx = set(CFG["mctx_contexts"])
    if len(ctx):
        held = ctx[(ctx["source"] == "cifar100c") | ~ctx["context"].isin(train_ctx)]
        per = held[held.kind.isin(["unl", "mctx"])].groupby(IDS)["renewal"].mean().reset_index()
        w6 = paired(per, "renewal")
        for m in METHODS:
            record("H6 multi-context reduces renewal (held-out contexts)", m, w6[w6.method == m]["diff"])
        print("held-out contexts:", sorted(set(held["context"])))
    if len(prim):
        w6s = paired(prim, "recovery")
        for m in METHODS:
            record("H6 multi-context reduces spontaneous recovery", m, w6s[w6s.method == m]["diff"])
    if len(ri):
        sib = ri[ri.condition == "sibling"]
        w6r = paired(sib, "recovery")
        for m in METHODS:
            record("H6 multi-context reduces reinstatement (sibling cues)", m, w6r[w6r.method == m]["diff"])
    mm = unl[(unl["kind"] == "mctx") & (unl["cls"] != PILOT)] if len(unl) else unl
    if len(mm):
        print("multi-context models, matched rate:", mm.groupby("method")["matched"].mean().round(3).to_dict())
else:
    print("No multi-context models yet (notebook 03 stage E, then notebook 04).")

## Bonus: semantic collateral damage (co-author's question)

The retraining gap G = retrained accuracy minus unlearned accuracy, on sibling classes vs unrelated classes.
Positive `G_related − G_unrelated` = unlearning damages semantic neighbours more than true retraining does.
Related prior work: "Retain-forget entanglement" (Cheng et al. 2026) and over-unlearning (Ha et al., ICML 2026).

In [ ]:
if len(unl_main):
    u = unl_main[unl_main["matched"] == True] if MATCHED_ONLY else unl_main
    u = u.assign(G_rel=u["ref_neighbour_test_acc"] - u["neighbour_test_acc"],
                 G_unrel=u["ref_unrelated_test_acc"] - u["unrelated_test_acc"])
    u["G_diff"] = u["G_rel"] - u["G_unrel"]
    for m in METHODS:
        record("Collateral (G_related - G_unrelated)", m, u[u.method == m]["G_diff"], family="control")
    display(u.groupby("method")[["G_rel", "G_unrel", "G_diff"]].mean().round(4))

## All tests with Holm correction

Holm correction is applied within each family (`main`, `control`, `robustness`). `significant` means
Holm-adjusted p < `ALPHA`. The table is saved to `tables/hypothesis_tests.csv`.

In [ ]:
res = pd.DataFrame(RESULTS)
if len(res):
    res["p_holm"] = np.nan
    for fam, idx in res.groupby("family").groups.items():
        sub = res.loc[idx]
        ok = sub["p_raw"].notna()
        if ok.sum():
            res.loc[sub.index[ok], "p_holm"] = multipletests(sub.loc[ok, "p_raw"], method="holm")[1]
    res["significant"] = res["p_holm"] < ALPHA
    res.to_csv(os.path.join(TAB_DIR, "hypothesis_tests.csv"), index=False)
    display(res.round(4))
else:
    print("No results to test yet.")

## Figures

1. **Extinction profile**: mean effect per method and main hypothesis (the headline figure).
2. **Spontaneous recovery curves**, one panel per proxy (primary BN mode).
3. **Renewal** per context.
4. **Reinstatement dose-response**: relatedness vs recovery.
5. **Savings curves** (relearning from `SAVINGS_SHOTS` images).
6. **Mechanism**: weight change per block, and CKA with the original model.
7. **BatchNorm controls**: recovery from statistics alone, and retain fine-tuning with BN updating vs frozen.
8. **Multi-context remedy**: standard vs multi-context models on held-out renewal and spontaneous recovery.

All figures are saved as PNG (300 dpi) and PDF in `figures/`, ready for the paper.

In [ ]:
def savefig(name):
    plt.tight_layout()
    for ext_ in ("png", "pdf"):
        plt.savefig(os.path.join(U.FIG_DIR, f"{name}.{ext_}"), dpi=300, bbox_inches="tight")
    plt.show()

# 1. extinction profile (main family)
main_res = res[res["family"] == "main"] if len(res) else res
if len(main_res):
    prof = main_res.pivot_table(index="method", columns="hypothesis", values="mean")
    fig, ax = plt.subplots(figsize=(1.6 * prof.shape[1] + 2, 0.6 * prof.shape[0] + 1.5))
    lim = np.nanmax(np.abs(prof.values)) if np.isfinite(np.nanmax(np.abs(prof.values))) and np.nanmax(np.abs(prof.values)) > 0 else 1
    im = ax.imshow(prof.values, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    ax.set_xticks(range(prof.shape[1])); ax.set_xticklabels(prof.columns, rotation=35, ha="right")
    ax.set_yticks(range(prof.shape[0])); ax.set_yticklabels(prof.index)
    for i in range(prof.shape[0]):
        for j in range(prof.shape[1]):
            v = prof.values[i, j]
            if np.isfinite(v): ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
    plt.colorbar(im, ax=ax, label="mean effect (positive = extinction-like / remedy works)")
    ax.set_title("Extinction profile of unlearning methods")
    savefig("fig1_extinction_profile")

In [ ]:
# 2. spontaneous recovery curves (primary BN mode, no BN-recalibration rows)
if len(ext) and "main_sr" in dir() and len(main_sr):
    d0 = main_sr[main_sr.kind.isin(["unl", "retrain"]) & (main_sr.proxy != "bn_recal")]
    proxies = list(dict.fromkeys(d0["proxy"]))
    fig, axes = plt.subplots(1, len(proxies), figsize=(4 * len(proxies), 3.2), squeeze=False)
    for ax, px in zip(axes[0], proxies):
        for m, g in d0[d0.proxy == px].groupby("method"):
            g = g.groupby("level")["recovery"].mean()
            ax.plot(g.index.astype(float), g.values, marker="o", label=m, lw=2 if m == "retrain" else 1.2,
                    color="black" if m == "retrain" else None, ls="--" if m == "retrain" else "-")
        ax.axhline(0, color="grey", lw=0.8); ax.set_title(px); ax.set_xlabel("level"); ax.set_ylabel(f"recovery ({METRIC})")
    axes[0][-1].legend(fontsize=7)
    savefig("fig2_spontaneous_recovery")

# 3. renewal by context
if len(ctx):
    t = ctx[ctx.kind.isin(["unl", "retrain"])].copy()
    t["label"] = t["context"] + np.where(t["severity"] > 0, "-s" + t["severity"].astype(str), "")
    pv = t.pivot_table(index="label", columns="method", values="renewal", aggfunc="mean")
    pv.plot(kind="bar", figsize=(max(6, 0.5 * len(pv)), 3.5), width=0.8)
    plt.axhline(0, color="grey", lw=0.8); plt.ylabel("renewal (context - clean)"); plt.title("Renewal (ABC)")
    savefig("fig3_renewal")

In [ ]:
# 4. reinstatement dose-response
if len(ri):
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for m, g in ri[ri.kind.isin(["unl", "retrain"])].groupby("method"):
        ax.scatter(g["relatedness"], g["recovery"], label=m, s=18, alpha=0.7, marker="x" if m == "retrain" else "o")
    ax.set_xscale("symlog", linthresh=1e-3); ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel("relatedness of cue classes (original model probability)"); ax.set_ylabel(f"recovery ({METRIC})")
    ax.legend(fontsize=7); ax.set_title(f"Reinstatement dose-response (BN {BN_PRIMARY})")
    savefig("fig4_reinstatement")

# 5. savings curves
if len(ext) and "sv_primary" in dir() and len(sv_primary):
    sv = sv_primary[sv_primary.kind.isin(["unl", "retrain"])]
    shots = sv["shots"].unique()
    s_use = SAVINGS_SHOTS if SAVINGS_SHOTS in shots else sorted(shots)[len(shots) // 2]
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for m, g in sv[sv.shots == s_use].groupby("method"):
        g = g.groupby("level")["forget_test_acc"].mean()
        ax.plot(g.index, g.values, marker="o", label=m, color="black" if m == "retrain" else None,
                ls="--" if m == "retrain" else "-")
    ax.set_xscale("symlog"); ax.set_xlabel("relearning steps"); ax.set_ylabel("forget-class accuracy")
    ax.set_title(f"Savings: relearning from {s_use} images (BN {BN_PRIMARY})"); ax.legend(fontsize=7)
    savefig("fig5_savings")

In [ ]:
# 6. mechanism: weight change and CKA
if len(mech):
    order = ["stem", "layer1", "layer2", "layer3", "layer4", "fc"]
    w = mech[(mech["test"] == "weights") & (mech.kind == "unl")]
    ck = mech[(mech["test"] == "cka") & (mech.kind == "unl") & (mech["ref"] == "original")]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    if len(w):
        pv = w.pivot_table(index="group", columns="method", values="rel_change", aggfunc="mean").reindex([o for o in order if o in set(w.group)])
        pv.plot(ax=axes[0], marker="o"); axes[0].set_ylabel("relative weight change vs original"); axes[0].set_title("Where unlearning changes the weights")
    if len(ck):
        blocks = ["stem", "layer1", "layer2", "layer3", "layer4", "penult", "logits"]
        colors = {m: f"C{i}" for i, m in enumerate(sorted(ck["method"].unique()))}   # one colour per method
        for (m, sub), g in ck.groupby(["method", "subset"]):
            g = g.groupby("block")["cka"].mean().reindex([b for b in blocks if b in set(g.block)])
            axes[1].plot(range(len(g)), g.values, marker="o", color=colors[m],
                         ls="-" if sub == "forget" else ":", label=f"{m} ({sub} images)")
        axes[1].set_xticks(range(len(blocks))); axes[1].set_xticklabels(blocks, rotation=30)
        axes[1].set_ylabel("CKA with original model"); axes[1].set_title("Representation similarity"); axes[1].legend(fontsize=6)
    savefig("fig6_mechanism")

In [ ]:
# 7. BatchNorm controls
if len(ext) and "sr" in dir() and (sr["proxy"] == "bn_recal").any():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    bnr = sr[(sr.proxy == "bn_recal") & sr.kind.isin(["unl", "retrain"])]
    for m, g in bnr.groupby("method"):
        g = g.groupby("level")["recovery"].mean()
        axes[0].plot(g.index, g.values, marker="o", label=m, color="black" if m == "retrain" else None,
                     ls="--" if m == "retrain" else "-")
    axes[0].set_xscale("log"); axes[0].axhline(0, color="grey", lw=0.8)
    axes[0].set_xlabel("retain images used to recompute BN statistics"); axes[0].set_ylabel(f"recovery ({METRIC})")
    axes[0].set_title("BatchNorm statistics alone (no weight change)"); axes[0].legend(fontsize=7)
    rf = sr[(sr.proxy == "retain_ft") & sr.kind.isin(["unl", "retrain"])]
    rf = rf[rf.level == rf.level.max()]
    pv = rf.pivot_table(index="method", columns="bn", values="recovery", aggfunc="mean")
    if len(pv):
        pv.plot(kind="bar", ax=axes[1], width=0.7)
        axes[1].axhline(0, color="grey", lw=0.8); axes[1].set_ylabel("recovery after retain fine-tuning")
        axes[1].set_title("Retain fine-tuning: BN updating vs frozen")
    savefig("fig7_batchnorm_controls")

# 8. multi-context remedy
if len(ext) and (ext["kind"] == "mctx").any():
    panels = []
    if "w6" in dir() and len(w6): panels.append(("renewal", w6, "held-out renewal"))
    if "w6s" in dir() and len(w6s): panels.append(("recovery", w6s, "spontaneous recovery (primary)"))
    if panels:
        fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 3.5), squeeze=False)
        for ax, (val, w, title) in zip(axes[0], panels):
            g = w.groupby("method")[["unl", "mctx"]].mean()
            g.rename(columns={"unl": "standard", "mctx": "multi-context"}).plot(kind="bar", ax=ax, width=0.7)
            ax.axhline(0, color="grey", lw=0.8); ax.set_ylabel(val); ax.set_title(title)
        savefig("fig8_multicontext_remedy")

## What to report

- `tables/hypothesis_tests.csv`: every test with family, n, mean, bootstrap CI, raw and Holm p-values. This is
  the main results table.
- `tables/H1_…csv`, `tables/H2a_…csv`: secondary tables per proxy/BN mode and per context.
- `figures/fig1…fig8` as PNG and PDF.

**Interpretation guide.**
- A method is *extinction-like* if it is matched (forgets as well as retraining at the output) and still shows
  significant positive effects in H1–H5, i.e. it recovers or keeps more than a model that never learned the
  class, and C1 (BatchNorm statistics alone) does not explain it.
- If C1 is large, or H1–H4 vanish when scored with the original statistics, the "recovery" may be the BatchNorm illusion. That is
  still a useful finding, but not extinction.
- A positive H6 means the theory also predicts a fix: multi-context unlearning relapses less.
- For the GroupNorm arm, run this notebook in the `-gn` copy with `norm="gn"`, `seeds=[0]` and the first 5
  classes. Agreement between the arms is the strongest evidence against a normalization artifact.

## Name of this run's output

Title this notebook **`ext-06-analysis`**. When the run has finished, create a dataset from its output with the
same name: open the notebook's viewer page → **Output** → **New Dataset** → title **`ext-06-analysis`**, Private.
(Or skip the dataset and attach the notebook output directly: *Add Input → Your Work → Notebooks*.)

| Output name | Contains | Attached by |
|---|---|---|
| `ext-06-analysis` | `tables/*.csv`, `figures/*.png|pdf` | nobody (final results for the paper) |

- **Re-runs and resumes:** don't create a new name. Add a **New Version** of the same dataset. Each output
  already contains everything from earlier runs (it's carried over), so only the latest version matters.
  In the notebooks that use it, update the input to the new version.
- **Smoke (dry) runs:** add `-smoke`, e.g. `ext-06-analysis-smoke`, so real and test outputs never share a dataset.
- **GroupNorm control arm** (`norm="gn"`): add `-gn`, e.g. `ext-06-analysis-gn`.